# itch.io Top Sellers — Analysis Notebook with Eight Charts

This notebook contains code for the eight requested charts. It analyses data collected by your group's scraper; it does **not** scrape itch.io itself.

**Important:** use your real collected data for final findings. Do not present demonstration or synthetic data as real results.

## 1. Imports and configuration

In [6]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("data/processed/cleaned_games.csv")
SNAPSHOT_DIR = Path("data/raw")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 110
print("Main data path:", DATA_PATH.resolve())

Main data path: C:\Users\Bienvenue\Desktop\cherry\data\processed\cleaned_games.csv


## 2. Load the collected dataset

In [9]:
def load_project_data():
    if DATA_PATH.exists():
        print(f"Loading {DATA_PATH}")
        return pd.read_csv(DATA_PATH)
    files = sorted(SNAPSHOT_DIR.glob("*.csv")) if SNAPSHOT_DIR.exists() else []
    if files:
        frames = []
        for file in files:
            frame = pd.read_csv(file)
            if not any(c.lower() in {"scraped_at", "snapshot_date", "date", "collected_at"} for c in frame.columns):
                match = re.search(r"(20\d{2}[-_]\d{2}[-_]\d{2})", file.stem)
                if match:
                    frame["snapshot_date"] = match.group(1).replace("_", "-")
            frames.append(frame)
        print(f"Combined {len(files)} snapshot files")
        return pd.concat(frames, ignore_index=True)
    raise FileNotFoundError(
        "No dataset found. Expected data/processed/cleaned_games.csv "
        "or daily CSV files in data/raw/."
    )

df = load_project_data()
print("Rows:", len(df), "| Columns:", len(df.columns))
display(df.head())
print(df.columns.tolist())

Loading data\processed\cleaned_games.csv
Rows: 15 | Columns: 10


,scraped_at,rank,title,url,developer,developer_url,price_text,genres,source_page,price_value
0,2026-10-15 13:02:52+00:00,1,GIF,https://puppetcombo.itch.io/nun-massacre,Puppet Combo,https://puppetcombo.itch.io,$4.95,Survival,https://itch.io/games/top-sellers,4.95
1,2026-10-15 13:02:52+00:00,2,GIF,https://zieelane.itch.io/helping-your-boyfriend,Zieelane,https://zieelane.itch.io,NaN,Visual Novel,https://itch.io/games/top-sellers,NaN
2,2026-10-15 13:02:52+00:00,3,GIF,https://wick.itch.io/lancer-tactics,Olive,https://wick.itch.io,$20,Strategy,https://itch.io/games/top-sellers,20.00
3,2026-10-15 13:02:52+00:00,4,GIF,https://elyaradine.itch.io/dressmaker,elyaradine,https://elyaradine.itch.io,NaN,Simulation,https://itch.io/games/top-sellers,NaN
4,2026-10-15 13:02:52+00:00,5,GIF,https://mortisfox.itch.io/john-doe-plus,Scopophobia Studios,https://mortisfox.itch.io,$1,Visual Novel,https://itch.io/games/top-sellers,1.00


['scraped_at', 'rank', 'title', 'url', 'developer', 'developer_url', 'price_text', 'genres', 'source_page', 'price_value']


## 3. Standardise columns and detect fields

In [10]:
def norm_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")

df.columns = [norm_name(c) for c in df.columns]
ALIASES = {
    "title": ["title", "game_title", "name", "product", "game"],
    "developer": ["developer", "creator", "author", "seller", "publisher"],
    "price": ["price_value", "price", "current_price", "sale_price", "displayed_price", "price_text"],
    "original_price": ["original_price", "regular_price", "list_price", "full_price"],
    "rank": ["rank", "ranking", "position", "top_seller_rank", "list_rank"],
    "date": ["scraped_at", "snapshot_date", "date", "collected_at", "collection_date", "timestamp"],
    "genre": ["genre", "genres", "tags", "tag", "category", "categories"],
    "platforms": ["platforms", "platform", "operating_systems", "os"],
}
def find_col(names):
    return next((n for n in names if n in df.columns), None)
COL = {k: find_col(v) for k, v in ALIASES.items()}
print("Detected fields:", COL)

def parse_money(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    if s == "free":
        return 0.0
    if s in {"", "name your own price", "pay what you want", "n/a", "na", "none"}:
        return np.nan
    s = s.replace(",", "")
    match = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(match.group()) if match else np.nan

df["_date"] = pd.to_datetime(df[COL["date"]], errors="coerce", utc=True).dt.tz_convert(None) if COL["date"] else pd.NaT
df["_price"] = df[COL["price"]].apply(parse_money) if COL["price"] else np.nan
df["_original_price"] = df[COL["original_price"]].apply(parse_money) if COL["original_price"] else np.nan
df["_rank"] = pd.to_numeric(df[COL["rank"]].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce") if COL["rank"] else np.nan
df["_title"] = df[COL["title"]].fillna("Unknown game").astype(str) if COL["title"] else "Unknown game"
df["_developer"] = df[COL["developer"]].fillna("Unknown developer").astype(str) if COL["developer"] else "Unknown developer"

display(pd.DataFrame({
    "check": ["rows", "unique titles", "missing prices", "missing ranks", "distinct snapshot dates"],
    "value": [len(df), df["_title"].nunique(), int(df["_price"].isna().sum()),
              int(df["_rank"].isna().sum()), int(df["_date"].dt.date.nunique()) if df["_date"].notna().any() else 0]
}))

Detected fields: {'title': 'title', 'developer': 'developer', 'price': 'price_value', 'original_price': None, 'rank': 'rank', 'date': 'scraped_at', 'genre': 'genres', 'platforms': None}


,check,value
0,rows,15
1,unique titles,1
2,missing prices,6
3,missing ranks,0
4,distinct snapshot dates,1


## 4. Chart helper functions

In [3]:
def finish_chart(fig, filename):
    fig.tight_layout()
    path = OUTPUT_DIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print("Saved:", path.resolve())

def split_multi(series):
    vals = []
    for x in series.dropna().astype(str):
        vals.extend(p.strip() for p in re.split(r"[,;|]", x) if p.strip())
    return pd.Series(vals, dtype="object")

## Chart 1 — Game price distribution (histogram and box plot)

In [4]:
prices = df["_price"].dropna()
if prices.empty:
    print("Chart 1 skipped: no numeric price values detected.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].hist(prices, bins=30, edgecolor="black")
    axes[0].set(title="Distribution of listed game prices", xlabel="Price (source currency)", ylabel="Number of observations")
    axes[0].grid(axis="y", alpha=.25)
    axes[1].boxplot(prices, showfliers=True)
    axes[1].set(title="Price spread and outliers", ylabel="Price (source currency)", xticks=[1], xticklabels=["Games"])
    axes[1].grid(axis="y", alpha=.25)
    finish_chart(fig, "chart1_price_distribution.png")
    print(f"Valid prices: {len(prices)} | Median: {prices.median():.2f} | Min: {prices.min():.2f} | Max: {prices.max():.2f}")

NameError: name 'df' is not defined

## Chart 2 — Games by genre (ranked bar chart)

In [ ]:
if not COL["genre"]:
    print("Chart 2 skipped: no genre/tags column. Update scraper or map the correct column.")
else:
    counts = split_multi(df[COL["genre"]]).value_counts().head(15).sort_values()
    if counts.empty:
        print("No genre/tag values were available.")
    else:
        fig, ax = plt.subplots(figsize=(10, 7))
        counts.plot(kind="barh", ax=ax)
        ax.set(title="Most frequently represented genres/tags", xlabel="Tag appearances", ylabel="Genre or tag")
        ax.grid(axis="x", alpha=.25)
        finish_chart(fig, "chart2_games_by_genre.png")
        print("A game can have multiple tags, so tag counts can exceed the number of games.")

## Chart 3 — Price versus Top Sellers rank (scatter plot)

In [ ]:
valid = df[["_price", "_rank"]].dropna()
if valid.empty:
    print("Chart 3 skipped: numeric price and rank values are required.")
else:
    fig, ax = plt.subplots()
    ax.scatter(valid["_rank"], valid["_price"], alpha=.55)
    ax.set(title="Listed price versus Top Sellers rank", xlabel="Rank (lower may mean higher position)", ylabel="Price (source currency)")
    ax.grid(alpha=.25)
    finish_chart(fig, "chart3_price_vs_rank.png")
    if len(valid) >= 3:
        print("Spearman correlation:", round(valid["_rank"].corr(valid["_price"], method="spearman"), 3))
    print("Rank is not a direct measurement of units sold or profit.")

## Chart 4 — Most represented developers (horizontal bar chart)

In [ ]:
counts = df["_developer"].replace("", "Unknown developer").value_counts()
counts = counts[counts.index != "Unknown developer"].head(15).sort_values()
if counts.empty:
    print("Chart 4 skipped: no developer names found.")
else:
    fig, ax = plt.subplots(figsize=(10, 7))
    counts.plot(kind="barh", ax=ax)
    ax.set(title="Developers with the most observed listings", xlabel="Listings/observations", ylabel="Developer / creator")
    ax.grid(axis="x", alpha=.25)
    finish_chart(fig, "chart4_top_developers.png")
    print("Repeated daily snapshots can count the same game more than once.")

## Chart 5 — Ranking changes over seven days (line chart)

In [ ]:
if not COL["title"] or df["_date"].notna().sum() == 0 or df["_rank"].notna().sum() == 0:
    print("Chart 5 skipped: needs title, rank, and date for each observation.")
else:
    tmp = df.dropna(subset=["_date", "_rank"]).copy()
    tmp["_day"] = tmp["_date"].dt.date
    days = sorted(tmp["_day"].unique())
    if len(days) < 2:
        print(f"Only {len(days)} distinct date(s) found. Collect daily snapshots first. Seven days are needed for the full assignment.")
    else:
        coverage = tmp.groupby("_title")["_day"].nunique().sort_values(ascending=False)
        selected = coverage.head(8).index
        pivot = tmp[tmp["_title"].isin(selected)].pivot_table(index="_day", columns="_title", values="_rank", aggfunc="mean").sort_index()
        fig, ax = plt.subplots(figsize=(12, 6))
        for title in pivot.columns:
            ax.plot(pivot.index.astype(str), pivot[title], marker="o", label=title[:32])
        ax.set(title="Daily Top Sellers rank movement", xlabel="Snapshot date", ylabel="Observed rank")
        ax.invert_yaxis()
        ax.tick_params(axis="x", rotation=35)
        ax.grid(alpha=.25)
        ax.legend(title="Game", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
        finish_chart(fig, "chart5_rank_changes.png")

## Chart 6 — Daily price trends (line chart)

In [ ]:
if df["_date"].notna().sum() == 0 or df["_price"].notna().sum() == 0:
    print("Chart 6 skipped: dates and numeric prices are required.")
else:
    tmp = df.dropna(subset=["_date", "_price"]).copy()
    tmp["_day"] = tmp["_date"].dt.date
    daily = tmp.groupby("_day")["_price"].agg(["median", "mean", "count"]).reset_index()
    if len(daily) < 2:
        print("Fewer than two dates found; a trend cannot yet be assessed.")
    else:
        fig, ax = plt.subplots()
        ax.plot(daily["_day"].astype(str), daily["median"], marker="o", label="Median price")
        ax.plot(daily["_day"].astype(str), daily["mean"], marker="s", label="Mean price")
        ax.set(title="Daily listed-price summary", xlabel="Snapshot date", ylabel="Price (source currency)")
        ax.tick_params(axis="x", rotation=35)
        ax.legend()
        ax.grid(alpha=.25)
        finish_chart(fig, "chart6_daily_price_trends.png")
        display(daily)
        print("The set of games can change daily; a median shift does not prove individual prices changed.")

## Chart 7 — Discount analysis (requires original and current prices)

In [ ]:
disc = df[["_price", "_original_price"]].dropna().copy()
disc = disc[(disc["_original_price"] > 0) & (disc["_price"] >= 0)]
if not COL["original_price"] or disc.empty:
    print("Chart 7 skipped: reliable original/list price and current price are both required. Do not infer discounts from one price.")
else:
    disc["discount"] = disc["_original_price"] > disc["_price"]
    counts = disc["discount"].value_counts().reindex([True, False], fill_value=0)
    fig, ax = plt.subplots()
    ax.bar(["Verifiable discount", "No verifiable discount"], [counts[True], counts[False]])
    ax.set(title="Listings with verifiable discounts", ylabel="Number of observations")
    ax.grid(axis="y", alpha=.25)
    finish_chart(fig, "chart7_discount_analysis.png")
    discounted = disc[disc["discount"]]
    if not discounted.empty:
        pct = (discounted["_original_price"] - discounted["_price"]) / discounted["_original_price"] * 100
        print(f"Median discount among discounted listings: {pct.median():.1f}%")
    print("Check that both prices use the same currency.")

## Chart 8 — Platform availability (bar chart)

In [ ]:
if not COL["platforms"]:
    print("Chart 8 skipped: no platforms/OS column was detected.")
else:
    counts = split_multi(df[COL["platforms"]]).value_counts().head(12).sort_values()
    if counts.empty:
        print("No platform values were available.")
    else:
        fig, ax = plt.subplots(figsize=(10, 6))
        counts.plot(kind="barh", ax=ax)
        ax.set(title="Platform availability among collected listings", xlabel="Platform mentions", ylabel="Platform")
        ax.grid(axis="x", alpha=.25)
        finish_chart(fig, "chart8_platform_availability.png")
        print("Platform mentions may be multiple per game; they are not exclusive categories.")

## Additional chart — Free versus paid listings

In [ ]:
if df["_price"].notna().any():
    group = pd.Series(np.where(df["_price"].isna(), "Unknown price",
                               np.where(df["_price"].eq(0), "Free / zero-priced", "Paid")),
                      index=df.index).value_counts()
    fig, ax = plt.subplots()
    group.plot(kind="bar", ax=ax)
    ax.set(title="Free/zero-priced versus paid listings", xlabel="Price group", ylabel="Number of observations")
    ax.tick_params(axis="x", rotation=0)
    ax.grid(axis="y", alpha=.25)
    finish_chart(fig, "additional_free_vs_paid.png")
    display(group.rename_axis("price_group").to_frame("observations"))
else:
    print("Price field unavailable.")

## 5. Save analysis-ready copy and summary

In [ ]:
df.to_csv(OUTPUT_DIR / "analysis_ready_copy.csv", index=False)
summary = {
    "observations": int(len(df)),
    "unique_titles": int(df["_title"].nunique()),
    "unique_developers": int(df["_developer"].nunique()),
    "distinct_snapshot_dates": int(df["_date"].dt.date.nunique()) if df["_date"].notna().any() else 0,
    "price_values_parsed": int(df["_price"].notna().sum()),
    "rank_values_parsed": int(df["_rank"].notna().sum()),
}
display(pd.DataFrame([summary]))
print("Saved outputs/analysis_ready_copy.csv")
print("Report only findings supported by your collected data.")
print("Keep raw daily files unchanged.")
print("Do not claim rank equals exact sales or profit.")
print("For the seven-day analysis, verify that seven distinct real snapshot dates exist.")